# 07 · Can LEMIS outcomes reach CITES records? The US overlap

**The question.** This is C5 in [`SYSTEM CODE/DESIGN_QUESTIONS.md`](../DESIGN_QUESTIONS.md). CITES has no enforcement labels, so how well WildGuard does on CITES records can only be shown indirectly. There is one exception worth testing. The US compiles its CITES annual report from the same import declarations LEMIS holds (notebook 03, X11), so for 2013–2014 the same US imports appear in both datasets: once as LEMIS line items with outcomes, once as CITES records the US reported as importer. If LEMIS outcomes can be attached to those CITES records, they form the only labelled set in CITES's own format, and the model can be scored on real CITES records for the one country where outcomes exist.

**Fixed before looking:**
1. **Populations.** LEMIS: line items of 2013–2014 (test years), animals with a class, species with a CITES listing in force. CITES: records of 2013–2014 that the US reported as importer (Appendix N already removed).
2. **Matching key.** LEMIS has no permit numbers, so records are matched on fields. The primary key is year, species, exporting country, product group, purpose and source. Notebook 05 (M4) used a looser key (year, species, exporting country), reported alongside.
3. **First measurement.** Is an enforcement case as likely to have a matching CITES record as a cleared line item? A refused or seized shipment may never have entered the US, and so may be missing from the US report.
4. **Rule.** The set is usable as labelled data if the coverage of enforcement cases and of cleared line items differ by less than 10 percentage points on the primary key. Otherwise the gap is reported as a finding about which labels can reach CITES at all, and the set is not used.
5. **If usable:** the unit is a matched key group (one CITES record can stand for several LEMIS line items and the reverse); its label is 1 if any matched LEMIS line item is an enforcement case; its score is the highest model score among its CITES records. The set is used for evaluation only, since 2013–2014 are test years.

In [1]:
import duckdb
import pandas as pd

import eda_setup  # noqa: F401  (puts SYSTEM CODE on the path)
from wildguard import config

con = duckdb.connect()
con.execute("PRAGMA threads=8")
con.execute(f"""
    CREATE VIEW lemis_us AS SELECT * FROM '{config.LEMIS_PARQUET}'
        WHERE year IN (2013, 2014) AND animal AND sp_listed;
    CREATE VIEW cites_us AS SELECT * FROM '{config.CITES_PARQUET}'
        WHERE year IN (2013, 2014) AND importer = 'US' AND reporter_type = 'I';
""")
KEYS = {"primary (year, species, exporter, product group, purpose, source)":
            ["year", "taxon", "exporter", "product_group", "purpose", "source"],
        "loose (year, species, exporter; notebook 05)": ["year", "taxon", "exporter"]}


def Q(sql):
    return con.execute(sql).df()


Q("""SELECT (SELECT count(*) FROM lemis_us) AS lemis_line_items,
            (SELECT sum(label) FROM lemis_us) AS lemis_cases,
            (SELECT count(*) FROM cites_us) AS cites_us_records""")

,lemis_line_items,lemis_cases,cites_us_records
0,234574,"4,744",246206


## 1. Coverage: does a LEMIS line item have a CITES record of the same trade?

In [2]:
rows = []
for name, key in KEYS.items():
    k = ", ".join(key)
    on = " AND ".join(f"l.{c} = c.{c}" for c in key)
    r = Q(f"""
        WITH c AS (SELECT DISTINCT {k} FROM cites_us)
        SELECT avg((c.year IS NOT NULL)::INT) FILTER (WHERE l.label = 1) AS cases_covered,
               avg((c.year IS NOT NULL)::INT) FILTER (WHERE l.label = 0) AS cleared_covered
        FROM lemis_us l LEFT JOIN c ON {on}""")
    rows.append({"key": name, **r.iloc[0].to_dict()})
cov = pd.DataFrame(rows).set_index("key")
cov["difference_points"] = 100 * (cov.cleared_covered - cov.cases_covered)
cov.style.format({"cases_covered": "{:.1%}", "cleared_covered": "{:.1%}", "difference_points": "{:.1f}"})

,cases_covered,cleared_covered,difference_points
key,,,
"primary (year, species, exporter, product group, purpose, source)",39.1%,72.0%,32.9
"loose (year, species, exporter; notebook 05)",86.1%,76.1%,-10.0


By outcome, on the primary key:

In [3]:
key = KEYS["primary (year, species, exporter, product group, purpose, source)"]
on = " AND ".join(f"l.{c} = c.{c}" for c in key)
Q(f"""
    WITH c AS (SELECT DISTINCT {", ".join(key)} FROM cites_us)
    SELECT CASE WHEN l.label = 0 THEN 'cleared'
                WHEN l.disposition = 'S' THEN 'seized'
                WHEN l.disposition = 'A' THEN 'abandoned'
                WHEN l.disposition = 'R' THEN 're-exported'
                ELSE 'refused, other disposition' END AS outcome,
           count(*) AS line_items, avg((c.year IS NOT NULL)::INT) AS covered
    FROM lemis_us l LEFT JOIN c ON {on}
    GROUP BY 1 ORDER BY 2 DESC""").set_index("outcome").style.format({"covered": "{:.1%}"})

,line_items,covered
outcome,,
cleared,229830,72.0%
seized,2633,31.3%
re-exported,1325,57.7%
abandoned,590,21.5%
"refused, other disposition",196,71.4%


## 2. The rule

In [4]:
gap = cov.loc[list(KEYS)[0], "difference_points"]
USABLE = abs(gap) < 10
print(f"Primary key: cleared coverage minus case coverage = {gap:.1f} points -> "
      f"{'usable as a labelled set' if USABLE else 'not usable as a labelled set'}")

Primary key: cleared coverage minus case coverage = 32.9 points -> not usable as a labelled set


## 3. What the matched groups would look like

Shown either way, so the size of what could be learned from this route is on record.

In [5]:
Q(f"""
    WITH l AS (SELECT {", ".join(key)}, count(*) AS line_items, max(label) AS any_case
               FROM lemis_us GROUP BY ALL),
         c AS (SELECT {", ".join(key)}, count(*) AS records FROM cites_us GROUP BY ALL)
    SELECT count(*) AS matched_groups, sum(l.any_case) AS groups_with_a_case,
           avg(l.any_case) AS share_with_a_case, sum(c.records) AS cites_records,
           sum(l.line_items) AS lemis_line_items
    FROM l JOIN c USING ({", ".join(key)})""")

,matched_groups,groups_with_a_case,share_with_a_case,cites_records,lemis_line_items
0,6925,768,0.1109,"168,587","167,349"


## Findings

| # | Finding | Consequence |
|---|---|---|
| U1 | On the primary key, 72.0% of cleared line items have a matching US-reported CITES record, but only 39.1% of enforcement cases: a gap of 32.9 points | The rule fails: the set is **not used** as labelled CITES data |
| U2 | The gap comes from the outcomes that stop a shipment: seized line items are covered 31%, abandoned 22%, re-exported 58%. The CITES report records trade that took place, and a seized or abandoned shipment did not enter the US | Most enforcement cases have no counterpart in CITES. A WildGuard flag on a CITES record means the record resembles trade that was stopped in the US, not that this record was stopped |
| U3 | On the loose key (year, species, exporter) cases are covered more often than cleared line items (86% against 76%): the species and routes of cases are reported, through other shipments of the same trade | What reaches CITES is the trade around a case, not the case itself |
| U4 | Had the set been usable, it would have held 6,925 matched groups, 768 of them with a case | — |

**Decision (C5):** no labelled set in CITES's own format. The recording shift (C4) is measured by the domain classifier on US-reported records only, and the evidence on CITES stays indirect (C7). U2 is stated in the report where it explains what a flag means (§1.4, §8).